<div style="background-color:#e6f2ff; padding:16px; border-radius:12px; border-left:6px solid #3b82f6;">
<h2 style="margin-top:0;">Project — NLP with Transformers on Airline Reviews</h2>
</div>

<div style="background-color:#e6f2ff; padding:16px; border-radius:12px; border-left:6px solid #3b82f6;">
<h2 style="margin-top:0;">Project Goals</h2>
<ul>
<li>Practice loading and exploring a real-world text dataset</li>
<li>Use a pretrained transformer model for sentiment or text classification</li>
<li>Connect transformer outputs to structured review variables such as rating and recommendation</li>
<li>Interpret NLP results in plain language</li>
<li>Think critically about what transformer models can and cannot tell us</li>
</ul>
</div>

## Context

Customer reviews are a rich source of text data. Airlines, hotels, universities, employers, and public agencies all collect large amounts of feedback in natural language. NLP helps researchers and analysts make sense of these texts at scale.

In this project, you will use `Airline_Reviews.csv` and a pretrained transformer model to analyze review sentiment and compare it to structured outcomes such as ratings and recommendations.

This is a useful social science style problem because it connects language to behavior, evaluation, and decision-making.

<div style="background-color:#e6f2ff; padding:16px; border-radius:12px; border-left:6px solid #3b82f6;">
<h2 style="margin-top:0;">Main Research Question</h2>
Can a transformer-based NLP model identify sentiment in airline reviews, and how well does that sentiment line up with customer ratings and recommendation decisions?
</div>

<div style="background-color:#e6f2ff; padding:16px; border-radius:12px; border-left:6px solid #3b82f6;">
<h2 style="margin-top:0;">Why use transformers here?</h2>
Transformers are modern NLP models that can capture context much better than simple word-count methods. For example, they are better at handling phrases where meaning depends on the surrounding words. In this project, we use a pretrained model so that students can focus on analysis and interpretation rather than model training.
</div>

<div style="background-color:#e6f2ff; padding:16px; border-radius:12px; border-left:6px solid #3b82f6;">
<h2 style="margin-top:0;">Which transformer should we use?</h2>
There are several reasonable choices for a project like this:
<ul>
<li><b>BERT</b>: a classic transformer family that is important historically and conceptually</li>
<li><b>RoBERTa</b>: a stronger training variant of BERT that often performs better on text classification tasks</li>
<li><b>DistilBERT</b>: a smaller, faster model that is often easier to use in classroom settings</li>
</ul>
We will use <code>distilbert-base-uncased-finetuned-sst-2-english</code>. It is fast, widely used for sentiment analysis, and easy to run through the Hugging Face pipeline interface. If you want a stronger modern alternative, a RoBERTa sentiment model is also a good option.
</div>

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

## 1. Setup

<div style="background-color:#e6f2ff; padding:16px; border-radius:12px; border-left:6px solid #3b82f6;">
<h2 style="margin-top:0;">Transformer setup</h2>
If `transformers` is not already installed in your environment, install it first. Depending on your setup, downloading a pretrained model may also require internet access the first time you run the notebook.
</div>

In [ ]:
# Uncomment if needed
# !pip install transformers torch sentencepiece

## 2. Load and Inspect the Dataset

<div style="background-color:#eaf7ea; padding:16px; border-radius:12px; border-left:6px solid #22c55e;">
<h2 style="margin-top:0;">Try It Yourself</h2>
Load `Airline_Reviews.csv` into a DataFrame called `df`. Then inspect the dataset with:
<ul>
<li>`df.head()`</li>
<li>`df.info()`</li>
<li>`df[['Overall_Rating', 'Recommended']].head()`</li>
</ul>
Answer:
<ul>
<li>How many rows are there?</li>
<li>Which column appears to contain the main review text?</li>
<li>Which column could be used as a structured outcome variable?</li>
</ul>
</div>

In [ ]:
# Load and inspect the dataset here


## 3. Explore Ratings and Recommendations

<div style="background-color:#eaf7ea; padding:16px; border-radius:12px; border-left:6px solid #22c55e;">
<h2 style="margin-top:0;">Try It Yourself</h2>
Explore the relationship between `Overall_Rating` and `Recommended`.
<br><br>
Create:
<ul>
<li>a count plot of `Recommended`</li>
<li>a histogram or bar plot of `Overall_Rating`</li>
</ul>
Then write 2 or 3 sentences describing how the reviews are distributed.
</div>

In [ ]:
# Explore recommendation and rating patterns here


## 4. Prepare the Text Data

<div style="background-color:#e6f2ff; padding:16px; border-radius:12px; border-left:6px solid #3b82f6;">
<h2 style="margin-top:0;">Why text preparation still matters</h2>
Transformers do much of their own tokenization internally, so you do not need to do the same kind of manual preprocessing you would do with older NLP methods. However, you still need to make sure that:
<ul>
<li>the text column is valid,</li>
<li>empty reviews are removed,</li>
<li>obvious formatting problems are handled,</li>
<li>you understand what rows you are analyzing.</li>
</ul>
A simple cleaning workflow for this project is:
<ol>
<li>keep the review column only if it is not missing,</li>
<li>convert reviews to strings,</li>
<li>strip extra whitespace,</li>
<li>remove rows where the review becomes empty,</li>
<li>optionally shorten very long text for quick experimentation.</li>
</ol>
</div>

<div style="background-color:#eaf7ea; padding:16px; border-radius:12px; border-left:6px solid #22c55e;">
<h2 style="margin-top:0;">Try It Yourself</h2>
Create a smaller DataFrame that keeps only rows with non-empty review text. Then select a manageable subset for experimentation, for example the first 200 or 500 reviews.
<br><br>
Hints:
<ul>
<li>Use <code>.dropna(subset=[...])</code> to remove missing reviews.</li>
<li>Use <code>.astype(str)</code> to ensure the column is treated as text.</li>
<li>Use <code>.str.strip()</code> to remove extra whitespace.</li>
<li>You can filter out empty strings with a boolean condition.</li>
</ul>
<br>
Display a few example review texts.
</div>

In [ ]:
# Prepare the review text here


## 5. Use a Pretrained Transformer Model

<div style="background-color:#e6f2ff; padding:16px; border-radius:12px; border-left:6px solid #3b82f6;">
<h2 style="margin-top:0;">A simple transformer workflow</h2>
The Hugging Face `pipeline` interface makes it easy to use pretrained transformer models. In this project, the simplest approach is to use a pretrained sentiment-analysis pipeline and apply it to each review.<br><br>
We will use <code>distilbert-base-uncased-finetuned-sst-2-english</code> as our main example. This allows you to ask: Does the model classify the review as positive or negative? And do those predictions match the reviewer's rating and recommendation?
</div>

In [ ]:
from transformers import pipeline

<div style="background-color:#eaf7ea; padding:16px; border-radius:12px; border-left:6px solid #22c55e;">
<h2 style="margin-top:0;">Try It Yourself</h2>
Create a sentiment-analysis pipeline using <code>distilbert-base-uncased-finetuned-sst-2-english</code> and run it on 5 example reviews first.
<br><br>
Look at the predicted label and confidence score for each review.
</div>

In [ ]:
# Recommended example:
# sentiment_model = pipeline(
#     "sentiment-analysis",
#     model="distilbert-base-uncased-finetuned-sst-2-english"
# )
# sentiment_model(list_of_reviews[:5])

# Your code here


## 6. Apply the Model to Many Reviews

<div style="background-color:#e6f2ff; padding:16px; border-radius:12px; border-left:6px solid #3b82f6;">
<h2 style="margin-top:0;">Why batch processing matters</h2>
Transformer models can be slow on large datasets. A practical workflow is to work with a subset first, then process reviews in batches. This is especially important when datasets are large, as in this airline review corpus.
</div>

<div style="background-color:#eaf7ea; padding:16px; border-radius:12px; border-left:6px solid #22c55e;">
<h2 style="margin-top:0;">Try It Yourself</h2>
Apply the sentiment model to a subset of reviews, for example 200 or 500 rows. Save the predicted label and confidence score into new columns such as:
<ul>
<li>`predicted_sentiment`</li>
<li>`sentiment_score`</li>
</ul>
</div>

In [ ]:
# Apply the model to a subset and store the results here


## 7. Compare Transformer Sentiment to Structured Review Measures

<div style="background-color:#e6f2ff; padding:16px; border-radius:12px; border-left:6px solid #3b82f6;">
<h2 style="margin-top:0;">What are we comparing?</h2>
Now the NLP results become analytically useful. We can compare transformer-based sentiment to:
<ul>
<li>`Overall_Rating`</li>
<li>`Recommended`</li>
</ul>
If the model is working reasonably well, positive sentiment should often align with high ratings and `yes` recommendations, while negative sentiment should align with low ratings and `no` recommendations.
</div>

<div style="background-color:#eaf7ea; padding:16px; border-radius:12px; border-left:6px solid #22c55e;">
<h2 style="margin-top:0;">Try It Yourself</h2>
Create at least two comparisons, such as:
<ul>
<li>average `Overall_Rating` by `predicted_sentiment`</li>
<li>a crosstab of `predicted_sentiment` and `Recommended`</li>
<li>a box plot of rating by predicted sentiment</li>
</ul>
Then write a short interpretation.
</div>

In [ ]:
# Compare sentiment outputs to rating and recommendation here


## 8. Inspect Misaligned Cases

<div style="background-color:#e6f2ff; padding:16px; border-radius:12px; border-left:6px solid #3b82f6;">
<h2 style="margin-top:0;">Why inspect mistakes?</h2>
NLP models often make mistakes in cases involving sarcasm, mixed opinions, domain-specific language, or long and complex reviews. Looking at mismatches between model predictions and structured outcomes helps you think critically about model limitations.
</div>

<div style="background-color:#eaf7ea; padding:16px; border-radius:12px; border-left:6px solid #22c55e;">
<h2 style="margin-top:0;">Try It Yourself</h2>
Find a few cases where the transformer prediction seems not to match the rating or recommendation.
<br><br>
For example, look for:
<ul>
<li>negative sentiment with `Recommended = yes`</li>
<li>positive sentiment with a low `Overall_Rating`</li>
</ul>
Read the review text and explain why the model may have struggled.
</div>

In [ ]:
# Inspect mismatched or ambiguous reviews here


## 9. Optional Extensions

<div style="background-color:#eaf7ea; padding:16px; border-radius:12px; border-left:6px solid #22c55e;">
<h2 style="margin-top:0;">Stretch Challenge</h2>
Try one or more of the following:
<ul>
<li>Use a zero-shot classification pipeline to label reviews by theme, such as `service`, `comfort`, `delay`, or `price`</li>
<li>Compare sentiment across airlines</li>
<li>Compare sentiment across traveler types or seat classes</li>
<li>Create a binary label from `Recommended` and evaluate how often sentiment aligns with it</li>
</ul>
</div>

In [ ]:
# Optional extension work here


## 11. Final Reflection

<div style="background-color:#eaf7ea; padding:16px; border-radius:12px; border-left:6px solid #22c55e;">

Write a short response that addresses the following:
<ol>
<li>What did the transformer model do well?</li>
<li>Where did it struggle?</li>
<li>How could airline companies or researchers use this kind of NLP analysis responsibly?</li>
<li>What are the risks of relying too heavily on automated sentiment analysis?</li>
</ol>
</div>

In [ ]:
# Write your final reflection here
